## Sieć neuronowa do rozpoznawania instrumentów

### Biblioteki

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

### Pobranie danych wejściowych z mel-spektogramu

In [ ]:
main = Path("../..").resolve()
spectrograms = main / Path("spectrograms/mels80_frame1024_step512_windowhann_sr44100_4s")
labels = spectrograms / "etykiety.csv"

data = pd.read_csv(labels)

In [5]:
print(data.shape)
print(data.columns.tolist())

(11011, 9)
['dataset', 'audio_file', 'Acoustic Guitar (steel)', 'Alto Sax', 'Bright Acoustic Piano', 'Drums', 'Electric Bass (finger)', 'Flute', 'String Ensemble 1']


In [7]:
print(list(spectrograms.iterdir()))

[PosixPath('/Users/khaczkiewicz/Documents/projekt_inzynierski/spectrograms/mels80_frame1024_step512_windowhann_sr44100_4s/test'), PosixPath('/Users/khaczkiewicz/Documents/projekt_inzynierski/spectrograms/mels80_frame1024_step512_windowhann_sr44100_4s/etykiety.csv'), PosixPath('/Users/khaczkiewicz/Documents/projekt_inzynierski/spectrograms/mels80_frame1024_step512_windowhann_sr44100_4s/train'), PosixPath('/Users/khaczkiewicz/Documents/projekt_inzynierski/spectrograms/mels80_frame1024_step512_windowhann_sr44100_4s/validation')]


In [17]:
instrumenty = [
    "Acoustic Guitar (steel)",
    "Alto Sax",
    "Bright Acoustic Piano",
    "Drums",
    "Electric Bass (finger)",
    "Flute",
    "String Ensemble 1"
]

In [18]:
for zbior in ["train", "validation", "test"]:
    folder = spectrograms / zbior
    pliki = list(folder.iterdir())

    print(f"\n{zbior}: {len(pliki)} elementów")
    print("Przykładowe pliki:", [p.name for p in pliki[:5]])


train: 7709 elementów
Przykładowe pliki: ['Track00986_32_9188.npy', 'Track01127_8_4707.npy', 'Track00849_72_10252.npy', 'Track01943_252_8239.npy', 'Track00343_100_5639.npy']

validation: 1646 elementów
Przykładowe pliki: ['Track01019_64_2140.npy', 'Track00849_72_5699.npy', 'Track01889_96_7835.npy', 'Track00343_128_9038.npy', 'Track01494_120_6376.npy']

test: 1656 elementów
Przykładowe pliki: ['Track00801_160_3815.npy', 'Track00343_192_10654.npy', 'Track02013_164_7697.npy', 'Track00343_80_8610.npy', 'Track00319_232_5610.npy']


In [19]:
# Sprawdzamy, czy nazwy plików NPY odpowiadają nazwom z CSV

for zbior in ["train", "validation", "test"]:
    pliki_npy = {
        p.stem
        for p in (spectrograms / zbior).glob("*.npy")
    }

    pliki_csv = set(
        data.loc[data["dataset"] == zbior, "audio_file"]
        .str.replace(".flac", "", regex=False)
    )

    print(f"\n{zbior}")
    print("Plików NPY:", len(pliki_npy))
    print("Rekordów CSV:", len(pliki_csv))
    print("Wspólnych nazw:", len(pliki_npy & pliki_csv))


train
Plików NPY: 7709
Rekordów CSV: 7709
Wspólnych nazw: 7709

validation
Plików NPY: 1646
Rekordów CSV: 1646
Wspólnych nazw: 1646

test
Plików NPY: 1656
Rekordów CSV: 1656
Wspólnych nazw: 1656


In [20]:
class InstrumentDataset(Dataset):
    def __init__(self, data, root, zbior):
        self.data = (
            data[data["dataset"] == zbior]
            .reset_index(drop=True)
        )
        self.root = root
        self.zbior = zbior

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        # Ścieżka do pliku spektrogramu
        nazwa_pliku = Path(row["audio_file"]).with_suffix(".npy").name
        sciezka = self.root / self.zbior / nazwa_pliku

        # Wczytanie mel-spektrogramu
        spec = np.load(sciezka)
        spec = torch.tensor(spec, dtype=torch.float32).unsqueeze(0)

        # Etykiety instrumentów
        label = row[instrumenty].to_numpy(dtype=np.float32)
        label = torch.tensor(label, dtype=torch.float32)

        return spec, label

In [21]:
train_dataset = InstrumentDataset(data, spectrograms, "train")
val_dataset = InstrumentDataset(data, spectrograms, "validation")
test_dataset = InstrumentDataset(data, spectrograms, "test")

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)

### Wybór urządzenia do obliczeń dla modelu

In [22]:
def get_device() -> torch.device:
    if torch.cuda.is_available():
        device = torch.device("cuda")
    elif torch.backends.mps.is_available():
        device = torch.device("mps")
    else:
        device = torch.device("cpu")

    return device

### Architektura sieci

In [23]:
# jest to wstępna, jak będą dane z mel-spektogramu to się rozbuduje sieć, aby jak najlepiej zoptymalizować ją

class Instrument_CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1,32,3,1)
        self.conv2 = nn.Conv2d(32,64,3,1)
        self.conv3 = nn.Conv2d(64,128,3,1)
        self.dropout2 = nn.Dropout2d(0.4)
        self.dropout3 = nn.Dropout2d(0.6)

        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))

        self.fc3 = nn.Linear(128, 7)

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)

        x = self.conv2(x)
        x = F.relu(x)
        x = F.max_pool2d(x, 2)
        x = self.dropout2(x)

        x = self.conv3(x)
        x = F.relu(x)
        x = F.max_pool2d(x, 2)
        x = self.dropout3(x)

        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        x = self.fc3(x)

        return x


In [24]:
device = get_device()

model = Instrument_CNN().to(device)

criterion = nn.MultiLabelSoftMarginLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

print("Urządzenie:", device)

Urządzenie: mps


In [25]:
print("Liczba próbek:", len(train_dataset))
print("Pierwszy rekord:")
print(train_dataset.data.iloc[0])

spec, label = train_dataset[0]

print("Kształt spektrogramu:", spec.shape)
print("Kształt etykiet:", label.shape)

Liczba próbek: 7709
Pierwszy rekord:
dataset                                      train
audio_file                 Track01881_24_1982.flac
Acoustic Guitar (steel)                          0
Alto Sax                                         0
Bright Acoustic Piano                            1
Drums                                            0
Electric Bass (finger)                           1
Flute                                            0
String Ensemble 1                                0
Name: 0, dtype: object
Kształt spektrogramu: torch.Size([1, 80, 346])
Kształt etykiet: torch.Size([7])


In [26]:
X, y = next(iter(train_loader))

print("Kształt spektrogramów:", X.shape)
print("Kształt etykiet:", y.shape)
print("Przykładowe etykiety:", y[0])

Kształt spektrogramów: torch.Size([4, 1, 80, 346])
Kształt etykiet: torch.Size([4, 7])
Przykładowe etykiety: tensor([0., 1., 0., 0., 1., 0., 0.])


In [27]:
epochs = 20

for epoch in range(epochs):
    model.train()
    train_loss = 0.0

    for X, y in train_loader:
        X = X.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        outputs = model(X)
        loss = criterion(outputs, y)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * X.size(0)

    train_loss /= len(train_loader.dataset)

    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for X, y in val_loader:
            X = X.to(device)
            y = y.to(device)

            outputs = model(X)
            loss = criterion(outputs, y)

            val_loss += loss.item() * X.size(0)

    val_loss /= len(val_loader.dataset)

    print(
        f"Epoka {epoch + 1}/{epochs} | "
        f"Train loss: {train_loss:.4f} | "
        f"Val loss: {val_loss:.4f}"
    )


Epoka 1/20 | Train loss: 0.6321 | Val loss: 0.5668
Epoka 2/20 | Train loss: 0.5716 | Val loss: 0.5282
Epoka 3/20 | Train loss: 0.5534 | Val loss: 0.5119
Epoka 4/20 | Train loss: 0.5382 | Val loss: 0.5089
Epoka 5/20 | Train loss: 0.5283 | Val loss: 0.4936
Epoka 6/20 | Train loss: 0.5200 | Val loss: 0.4992
Epoka 7/20 | Train loss: 0.5135 | Val loss: 0.4782
Epoka 8/20 | Train loss: 0.5079 | Val loss: 0.4716
Epoka 9/20 | Train loss: 0.5033 | Val loss: 0.4704
Epoka 10/20 | Train loss: 0.4969 | Val loss: 0.4692
Epoka 11/20 | Train loss: 0.4924 | Val loss: 0.4568
Epoka 12/20 | Train loss: 0.4898 | Val loss: 0.4570
Epoka 13/20 | Train loss: 0.4831 | Val loss: 0.4455
Epoka 14/20 | Train loss: 0.4813 | Val loss: 0.4387
Epoka 15/20 | Train loss: 0.4802 | Val loss: 0.4460
Epoka 16/20 | Train loss: 0.4746 | Val loss: 0.4377
Epoka 17/20 | Train loss: 0.4697 | Val loss: 0.4341
Epoka 18/20 | Train loss: 0.4684 | Val loss: 0.4462
Epoka 19/20 | Train loss: 0.4627 | Val loss: 0.4298
Epoka 20/20 | Train l

In [28]:
model.eval()

with torch.no_grad():
    for X, y in test_loader:
        X = X.to(device)
        y = y.to(device)

        logits = model(X)
        prawdopodobienstwa = torch.sigmoid(logits)
        predykcje = (prawdopodobienstwa >= 0.5).float()

        print("Prawdopodobieństwa:")
        print(prawdopodobienstwa.cpu())

        print("\nPrzewidywane etykiety:")
        print(predykcje.cpu())

        print("\nPrawdziwe etykiety:")
        print(y.cpu())

        break


Prawdopodobieństwa:
tensor([[0.6549, 0.4510, 0.5624, 0.9925, 0.6795, 0.4762, 0.6351],
        [0.5753, 0.4208, 0.6865, 1.0000, 0.1127, 0.4025, 0.5122],
        [0.0572, 0.5534, 0.0761, 1.0000, 0.0069, 0.5352, 0.2319],
        [0.3400, 0.4606, 0.4882, 1.0000, 0.0672, 0.5368, 0.7219]])

Przewidywane etykiety:
tensor([[1., 0., 1., 1., 1., 0., 1.],
        [1., 0., 1., 1., 0., 0., 1.],
        [0., 1., 0., 1., 0., 1., 0.],
        [0., 0., 0., 1., 0., 1., 1.]])

Prawdziwe etykiety:
tensor([[1., 0., 1., 1., 1., 1., 1.],
        [0., 1., 1., 1., 0., 0., 0.],
        [0., 1., 0., 1., 0., 1., 0.],
        [0., 1., 0., 1., 0., 0., 1.]])
